# Vantage tuning: Kaggle 2x T4 (tuning v4)

The baseline run (`vantage_t4_baseline_v5.ipynb`) measured where latency breaks. This run explains why and
tries to move it. Vantage starts vLLM with one setting changed at a time, drives the same load
points, and scrapes vLLM's `/metrics` every second: KV-cache use, requests waiting inside the
engine, preemptions, prefix-cache hits.

| server | change | question |
|---|---|---|
| `base` | none (baseline settings), concurrency 1 to 64 | where does the KV cache fill, and does that match the knee? |
| `noprefix` | prefix caching off | what is reusing the shared system prompt worth? |
| `mem95` | `--gpu-memory-utilization 0.95` | does more KV cache move the knee? |
| `seqs32` | `--max-num-seqs 32` | does capping the batch protect latency past the knee? |
| `chunk512` | `--max-num-batched-tokens 512` | do smaller prefill chunks cut inter-token latency spikes? |
| `awq` | AWQ int4 on T4 #1, concurrency 1 to 64 | why AWQ is faster or slower than fp16 here |

Every load point starts with a **cold prefix cache**: baseline v3 drew each point's prompts in the
same order, so a point re-sent the prompts the previous one had just cached (99% hits at AWQ c=2) and
its first-token times were flattered. This run's `base` and `awq` curves replace those numbers.
Then **the cross-check**: `vllm bench serve` against the same fp16 server, compared with the `base`
points at concurrency 1, 8 and 32.

All settings are in `configs/sweeps/kaggle-t4-tune.yaml`; the exact command of every server is
written to `servers.json`.

**To run:** Accelerator **GPU T4 x2**, Internet **on**, then **Save Version, Save & Run All
(Commit)**. About 2.5 hours. Send me `vantage_t4_tuning_v4_results.zip`.

## 1. Settings and helpers

In [ ]:
import datetime, glob, json, os, shutil, subprocess, sys, time, traceback, urllib.error, urllib.request

NOTEBOOK = "tune"                              # "base" or "tune"; set by build_kaggle.py
DRY_RUN = os.environ.get("VANTAGE_DRY_RUN") == "1"     # no GPUs: simulator / fake server (testing)
WORK = os.environ.get("VANTAGE_WORK", "/kaggle/working")
REPO_URL = os.environ.get("VANTAGE_REPO_URL", "https://github.com/SiddharthaRaja/vantage.git")
# Not ".../vantage": a folder named like the package in the working directory
# shadows the installed package when Python looks there first.
REPO = f"{WORK}/vantage_repo"
RUN = ("kaggle_t4_tune_" if NOTEBOOK == "tune" else "kaggle_t4_") + datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d_%H%M")
OUT = f"{WORK}/{RUN}"
os.makedirs(OUT, exist_ok=True)

FP16 = "Qwen/Qwen3-4B-Instruct-2507"
AWQ = "cyankiwi/Qwen3-4B-Instruct-2507-AWQ-4bit"
# First one that serves on a T4 wins. Older vLLM needs the transformers of its
# day (0.10.2 fails on today's Qwen2Tokenizer, 0.9.2 on a duplicate config name).
VLLM_CANDIDATES = ["vllm", "vllm==0.10.2 transformers==4.56.2", "vllm==0.9.2 transformers==4.53.3"]
CROSSCHECK = [(1, 48), (8, 96), (32, 160)]                  # (concurrency, requests), as in the sweep

# No VLLM_API_KEY: vLLM reads that variable and switches on bearer auth, so a
# key set "for the client" made every server answer 401 (baseline v2 lost an
# hour waiting on a server that was up). The client sends "EMPTY", which an
# unauthenticated vLLM ignores.
os.environ.pop("VLLM_API_KEY", None)
os.environ.update(PYTHONUNBUFFERED="1", GPU_HOURLY_USD=os.environ.get("GPU_HOURLY_USD", "0.35"))
STEPS = {}
T0 = time.time()


def sh(cmd, log=None, timeout=None, cwd=None, env=None):
    """Run a shell command, stream it to a log file, return the exit code (never raises)."""
    log = log or f"{OUT}/shell.log"
    with open(log, "a") as fh:
        fh.write(f"\n$ {cmd}\n"); fh.flush()
        try:
            p = subprocess.run(cmd, shell=True, cwd=cwd, env=env, stdout=fh,
                               stderr=subprocess.STDOUT, timeout=timeout)
            return p.returncode
        except subprocess.TimeoutExpired:
            fh.write(f"TIMEOUT after {timeout}s\n")
            return 124


def tail(path, n=25):
    try:
        print("".join(open(path, errors="replace").readlines()[-n:]))
    except OSError:
        print(f"(no {path})")


def run_step(name, fn, needs=()):
    """Run one step; record ok / failed / skipped; never raise."""
    missing = [d for d in needs if STEPS.get(d, {}).get("status") != "ok"]
    t = time.time()
    if missing:
        STEPS[name] = {"status": "skipped", "why": f"needs {', '.join(missing)}"}
    else:
        try:
            note = fn()
            STEPS[name] = {"status": "ok", "note": note, "minutes": round((time.time() - t) / 60, 1)}
        except Exception as e:  # noqa: BLE001
            STEPS[name] = {"status": "failed", "error": f"{type(e).__name__}: {e}",
                           "minutes": round((time.time() - t) / 60, 1)}
            traceback.print_exc()
    json.dump(STEPS, open(f"{OUT}/steps.json", "w"), indent=2, default=str)
    print(f"[{name}] {STEPS[name]['status']}  {STEPS[name].get('note') or STEPS[name].get('error') or STEPS[name].get('why') or ''}")


print("run ->", OUT, "| dry run" if DRY_RUN else "")

## 2. GPUs, Vantage, workload

In [ ]:
def setup():
    if not DRY_RUN:
        sh("nvidia-smi", log=f"{OUT}/nvidia_smi.txt")
        n = subprocess.run("nvidia-smi -L | wc -l", shell=True, capture_output=True, text=True).stdout.strip()
        if n != "2":
            raise RuntimeError(f"expected 2 GPUs, found {n}: set Accelerator to GPU T4 x2")
    if os.path.exists(REPO):
        shutil.rmtree(REPO)
    if sh(f"git clone -q {REPO_URL} {REPO}") != 0:
        raise RuntimeError("git clone failed")
    sh("git log --oneline -3", log=f"{OUT}/commit.txt", cwd=REPO)
    if sh(f"{sys.executable} -m pip install -q -e {REPO}", timeout=900) != 0:
        raise RuntimeError("pip install of Vantage failed (see shell.log)")
    for f in ["data/workload.vectora.jsonl", "data/eval.references.jsonl"]:
        if not os.path.exists(f"{REPO}/{f}"):
            raise RuntimeError(f"missing {f}")
    sh("vantage workload data/workload.vectora.jsonl", log=f"{OUT}/workload_shape.txt", cwd=REPO)
    commits = [l for l in open(f"{OUT}/commit.txt").read().splitlines() if l.strip() and not l.startswith("$")]
    return commits[0][:70] if commits else "cloned"

run_step("setup", setup)

## 3. vLLM

vLLM goes into **its own Python 3.12 environment** (made with `uv`), not Kaggle's. Kaggle's image is
Python 3.13 with hundreds of preinstalled packages; installing vLLM into it once crashed vLLM on
start-up inside `transformers`' audio imports, and older vLLM versions have no Python 3.13 builds at
all. Vantage only talks to vLLM over HTTP, so the two never need to share packages. The
environment's `bin` goes first on `PATH`, so every `vllm` command below is the isolated one.

The newest vLLM is tried first; if it cannot serve on a T4 (compute capability 7.5), the next
candidate is installed into a fresh environment in step 4. The version used goes into the report.

In [ ]:
VLLM_VERSION = None

VENV = "/tmp/vllm-venv"     # outside /kaggle/working, so it is not saved as output

def install_vllm(spec):
    """A fresh, isolated Python 3.12 environment with `spec` installed; its bin first on PATH."""
    global VLLM_VERSION
    if DRY_RUN:
        VLLM_VERSION = "dry-run"; return VLLM_VERSION
    log = f"{OUT}/vllm_install.log"
    if sh(f"{sys.executable} -m pip install -q uv", log=log, timeout=600) != 0:
        raise RuntimeError("pip install uv failed")
    shutil.rmtree(VENV, ignore_errors=True)
    if sh(f"{sys.executable} -m uv venv -q --python 3.12 {VENV}", log=log, timeout=900) != 0:
        raise RuntimeError("could not create the Python 3.12 environment (see vllm_install.log)")
    if sh(f"{sys.executable} -m uv pip install -q --python {VENV}/bin/python {spec}", log=log, timeout=1800) != 0:
        raise RuntimeError(f"installing {spec} failed (see vllm_install.log)")
    # vllm bench serve needs the bench extras (pandas, datasets) for --dataset-name custom
    sh(f"{sys.executable} -m uv pip install -q --python {VENV}/bin/python pandas datasets", log=log, timeout=900)
    if not os.environ["PATH"].startswith(f"{VENV}/bin:"):
        os.environ["PATH"] = f"{VENV}/bin:" + os.environ["PATH"]
    VLLM_VERSION = subprocess.run([f"{VENV}/bin/python", "-c", "import vllm; print(vllm.__version__)"],
                                  capture_output=True, text=True).stdout.strip()
    if not VLLM_VERSION:
        raise RuntimeError(f"{spec} installed but does not import (see vllm_install.log)")
    sh(f"{VENV}/bin/python -c 'import sys, torch; print(sys.version); print(torch.__version__, torch.cuda.is_available())'",
       log=log)
    return f"vllm {VLLM_VERSION} (isolated Python 3.12)"

run_step("install_vllm", lambda: install_vllm(VLLM_CANDIDATES[0]), needs=["setup"])

## 4. Pre-flight: which vLLM serves on a T4?

Starts the fp16 server once, with the same fallbacks as the baseline run (`--enforce-eager`, then
older vLLM versions), and stops it again. Whatever flag it needed is passed to every server in the
tuning sweep through `VLLM_EXTRA`, so all variants differ only in the setting under test.

In [ ]:
SERVERS = {}
VLLM_EXTRA = ""

def _up(port):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{port}/v1/models", timeout=3)
        return True
    except urllib.error.HTTPError as e:
        if e.code in (401, 403):     # up, but refusing us: say so instead of waiting 25 minutes
            raise RuntimeError(f"server on port {port} answers {e.code}: it was started with an API key")
        return False
    except Exception:
        return False


def launch(model, gpu, port, extra=()):
    # dev mode exposes /reset_prefix_cache, which Vantage calls before each load point
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "VLLM_SERVER_DEV_MODE": "1"}
    log = open(f"{OUT}/server_{port}.log", "a")
    cmd = ["vllm", "serve", model, "--dtype", "half", "--max-model-len", "4096",
           "--gpu-memory-utilization", "0.90", "--enable-prefix-caching", "--port", str(port), *extra]
    log.write("\n$ " + " ".join(cmd) + "\n"); log.flush()
    SERVERS[port] = subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT)


def wait_up(ports, limit_s=1500):
    t = time.time()
    while time.time() - t < limit_s:
        if all(_up(p) for p in ports):
            return True
        if any(SERVERS[p].poll() is not None for p in ports):
            return False
        time.sleep(10)
    return False


def stop_servers():
    for p in SERVERS.values():
        p.terminate()
    time.sleep(10)
    for p in SERVERS.values():
        if p.poll() is None:
            p.kill()
    SERVERS.clear()
    sh("pkill -f 'vllm serve' || true")
    time.sleep(5)


def start_servers(models=((None, 0, 8000), (None, 1, 8001))):
    """Start one server per (model, gpu, port); fall back until one vLLM version serves."""
    global VLLM_EXTRA
    models = [(m or (FP16 if gpu == 0 else AWQ), gpu, port) for m, gpu, port in models]
    if DRY_RUN:
        VLLM_EXTRA = ""
        return "simulator"
    attempts = [(VLLM_CANDIDATES[0], ())] + [(VLLM_CANDIDATES[0], ("--enforce-eager",))] + \
               [(v, ("--enforce-eager",)) for v in VLLM_CANDIDATES[1:]]
    current = VLLM_CANDIDATES[0]
    for spec, extra in attempts:
        if spec != current:
            current = spec
            try:
                install_vllm(spec)
            except RuntimeError as e:
                print(f"{spec}: {e}")
                continue
        for model, gpu, port in models:
            launch(model, gpu, port, extra)
        ports = [port for _, _, port in models]
        if wait_up(ports):
            VLLM_EXTRA = " ".join(extra)
            return f"vllm {VLLM_VERSION} {VLLM_EXTRA or '(cuda graphs on)'}"
        print(f"servers did not start with {spec} {extra}; last lines:")
        for port in ports:
            tail(f"{OUT}/server_{port}.log", 15)
        stop_servers()
    raise RuntimeError("no vLLM version served on the T4s (see server_*.log)")

In [ ]:
def preflight():
    note = start_servers(models=[(FP16, 0, 8000)])
    stop_servers()
    os.environ["VLLM_EXTRA"] = VLLM_EXTRA        # read by ${VLLM_EXTRA:-} in kaggle-t4-tune.yaml
    return note

run_step("preflight", preflight, needs=["install_vllm"])

## 6. The sweeps

Load points run one at a time. Raw per-request records, charts and `REPORT.md` are copied into the
run folder as each sweep finishes.

In [ ]:
SIM = {"vllm.qwen3-4b-2507-awq-t4": "sim.int4", "vllm.qwen3-4b-2507-t4": "sim.fp16"}

FAKE = {"vllm serve Qwen/Qwen3-4B-Instruct-2507": f"{sys.executable} {REPO}/tests/fake_vllm.py --model Qwen/Qwen3-4B-Instruct-2507",
        "vllm serve cyankiwi/Qwen3-4B-Instruct-2507-AWQ-4bit": f"{sys.executable} {REPO}/tests/fake_vllm.py --model cyankiwi/Qwen3-4B-Instruct-2507-AWQ-4bit"}

def sweep_config(name):
    path = f"{REPO}/configs/sweeps/{name}.yaml"
    if not DRY_RUN:
        return path
    text = open(path).read()                       # dry run: no GPU, tiny loads
    if "servers:" in text:                         # managed servers: a fake vLLM that Vantage starts
        for real, fake in FAKE.items():
            text = text.replace(real, fake)
    else:                                          # servers started by the notebook: the simulator
        for real, sim in SIM.items():
            text = text.replace(real, sim)
    for a, b in [("n_requests: 160", "n_requests: 12"), ("n_requests: 96", "n_requests: 8"),
                 ("n_requests: 48", "n_requests: 4"), ("n_requests: 30", "n_requests: 3"),
                 ("duration_s: 60", "duration_s: 2"), ("cooldown_s: 10", "cooldown_s: 0"),
                 ("cooldown_s: 5", "cooldown_s: 0"), ("[0.25, 0.5]", "[2, 4]")]:
        text = text.replace(a, b)
    os.makedirs(f"{OUT}/dry_configs", exist_ok=True)
    open(f"{OUT}/dry_configs/{name}.yaml", "w").write(text)
    return f"{OUT}/dry_configs/{name}.yaml"


def sweep(name, timeout_h=4):
    log = f"{OUT}/{name}.log"
    rc = sh(f"vantage sweep -c {sweep_config(name)}", log=log, cwd=REPO, timeout=int(timeout_h * 3600))
    runs = sorted(glob.glob(f"{REPO}/results/{name}/*"))
    if runs:
        shutil.copytree(runs[-1], f"{OUT}/{name}", dirs_exist_ok=True)
    tail(log, 12)
    if rc != 0 or not os.path.exists(f"{OUT}/{name}/REPORT.md"):
        raise RuntimeError(f"sweep exited {rc} (see {name}.log)")
    gate = f"{OUT}/{name}/quality_gate.json"
    if os.path.exists(gate):
        g = json.load(open(gate))[0]
        return f"gate {g['verdict']}: delta {g['delta']:+.3f}, 95% CI {g['delta_ci95']}"
    return "done"

In [ ]:
run_step("kaggle-t4-tune", lambda: sweep("kaggle-t4-tune", timeout_h=5), needs=["preflight"])

## 7. Cross-check against vLLM's own benchmark

Same server (fp16, port 8000), same input tokens: `scripts/vllm_bench_dataset.py` applies the model's
chat template, so `vllm bench serve` sends exactly what Vantage sends. Compared at concurrency 1, 8
and 32 with the request counts the sweep used. Agreement within ~10% on TTFT and inter-token latency
means Vantage measures what vLLM measures; a bigger gap is a finding to explain. In the tuning
run the sweep has stopped its servers, so the fp16 server is started once more with the baseline
settings and compared with the sweep's `base` points; the prefix cache is cleared before each run.

In [ ]:
BENCH_FLAGS = [
    ["--custom-output-len", "192", "--skip-chat-template"],
    ["--custom-output-len", "192", "--custom-skip-chat-template"],
    ["--skip-chat-template"],
]

def vllm_bench(c, n):
    out = f"{OUT}/vllm-bench"
    os.makedirs(out, exist_ok=True)
    base = (f"vllm bench serve --backend vllm --model {FP16} --base-url http://127.0.0.1:8000 "
            f"--endpoint /v1/completions --dataset-name custom --dataset-path data/vllm_bench.vectora.jsonl "
            f"--num-prompts {n} --max-concurrency {c} --save-result --result-dir {out} --result-filename c{c}.json")
    for flags in BENCH_FLAGS:
        if sh(base + " " + " ".join(flags), log=f"{OUT}/vllm_bench.log", cwd=REPO, timeout=3600) == 0:
            return json.load(open(f"{out}/c{c}.json")), flags
    raise RuntimeError("vllm bench serve failed with every flag set (see vllm_bench.log)")


def reset_cache(port=8000):
    try:
        urllib.request.urlopen(urllib.request.Request(f"http://127.0.0.1:{port}/reset_prefix_cache", method="POST"), timeout=30)
        return True
    except Exception:
        return False


def crosscheck(sweep_name="kaggle-t4", group="fp16"):
    if DRY_RUN:
        return "skipped in dry run"
    rc = sh(f"{sys.executable} scripts/vllm_bench_dataset.py --model {FP16}", log=f"{OUT}/vllm_bench.log", cwd=REPO)
    if rc != 0:                                    # Kaggle's transformers failed: try the vLLM environment's
        sh(f"{sys.executable} -m uv pip install -q --python {VENV}/bin/python pyyaml httpx pydantic",
           log=f"{OUT}/vllm_bench.log")
        rc = sh(f"{VENV}/bin/python scripts/vllm_bench_dataset.py --model {FP16}", log=f"{OUT}/vllm_bench.log", cwd=REPO)
    if rc != 0:
        raise RuntimeError("could not build the vllm bench dataset")
    ours = {s["config_id"]: s for s in json.load(open(f"{OUT}/{sweep_name}/summaries.json"))}
    rows = ["| concurrency | metric | Vantage | vllm bench | Vantage / vllm |", "|---|---|---|---|---|"]
    used = None
    for c, n in CROSSCHECK:
        reset_cache()                     # the same cold start Vantage's points get
        v, used = vllm_bench(c, n)
        s = ours.get(f"{group}[concurrency={c}]")
        if not s:
            continue
        ms = lambda x: x * 1000 if x is not None else None  # noqa: E731
        num = lambda x: f"{x:,.1f}" if x is not None else "n/a"  # noqa: E731
        # Vantage's "ITL" is each request's mean gap between tokens, which is
        # what vLLM calls TPOT; vLLM's ITL pools every individual gap, a
        # different statistic (they agree only when nothing interrupts decoding).
        pairs = [("TTFT p50 (ms)", ms(s["ttft"]["p50"]), v.get("median_ttft_ms")),
                 ("TTFT p99 (ms)", ms(s["ttft"]["p99"]), v.get("p99_ttft_ms")),
                 ("per-token latency p50 (ms) vs TPOT", ms(s["itl"]["p50"]), v.get("median_tpot_ms")),
                 ("output tok/s", s["output_tokens_per_s"], v.get("output_throughput"))]
        for metric, a, b in pairs:
            ratio = f"{a / b:.2f}" if a and b else "n/a"
            rows.append(f"| {c} | {metric} | {num(a)} | {num(b)} | {ratio} |")
    text = (f"# Cross-check: Vantage vs `vllm bench serve` (vLLM {VLLM_VERSION}, flags {used})\n\n"
            "Same server, same input tokens. Vantage's per-token latency is each request's mean gap "
            "between tokens, compared with vLLM's TPOT (the same definition). TTFT p99 is the single "
            "worst request or two at these sample sizes, and vllm bench has no warm-up, so tails "
            "differ more than medians.\n\n" + "\n".join(rows) + "\n")
    open(f"{OUT}/CROSSCHECK.md", "w").write(text)
    print(text)
    return "written"

In [ ]:
def crosscheck_tune():
    start_servers(models=[(FP16, 0, 8000)])
    try:
        return crosscheck("kaggle-t4-tune", "base")
    finally:
        stop_servers()

run_step("crosscheck", crosscheck_tune, needs=["kaggle-t4-tune"])

## 9. Summary and the zip

In [ ]:
def package():
    if SERVERS:
        stop_servers()
    sh("nvidia-smi", log=f"{OUT}/nvidia_smi_end.txt")
    title = "Vantage tuning on Kaggle 2x T4" if NOTEBOOK == "tune" else "Vantage on Kaggle 2x T4"
    parts = [f"# {title} ({RUN})", "",
             f"vLLM {VLLM_VERSION}; total {(time.time() - T0) / 3600:.2f} h; dry run: {DRY_RUN}", "",
             "| step | status | note |", "|---|---|---|"]
    for k, v in STEPS.items():
        parts.append(f"| {k} | {v['status']} | {v.get('note') or v.get('error') or v.get('why') or ''} |")
    for f in ["sanity_check.txt"]:
        if os.path.exists(f"{OUT}/{f}"):
            parts += ["", "## Sanity check", "", "```", open(f"{OUT}/{f}").read()[:3000], "```"]
    for name in ["kaggle-t4", "kaggle-t4-offered-load", "kaggle-t4-tune"]:
        if os.path.exists(f"{OUT}/{name}/REPORT.md"):
            parts += ["", "---", "", open(f"{OUT}/{name}/REPORT.md").read()]
    if os.path.exists(f"{OUT}/CROSSCHECK.md"):
        parts += ["", "---", "", open(f"{OUT}/CROSSCHECK.md").read()]
    if os.path.exists(f"{OUT}/cache_sim.log"):
        parts += ["", "---", "", "# Semantic cache", "", "```", open(f"{OUT}/cache_sim.log").read()[-3000:], "```"]
    open(f"{OUT}/SUMMARY.md", "w").write("\n".join(parts) + "\n")
    stem = "vantage_t4_tuning_v4_results" if NOTEBOOK == "tune" else "vantage_t4_baseline_v5_results"
    zip_path = shutil.make_archive(f"{WORK}/{stem}", "zip", WORK, RUN)
    return f"{zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)"

run_step("package", package)
print(open(f"{OUT}/SUMMARY.md").read()[:4000])